# 04 · Analysis — Diagnostic (Path A) + Predictive (Path B)

**Stages 5a → 4 → 5b → 6 → Cross-Cutting of [STRUCTURE.md](../DOCS/STRUCTURE.md).** This notebook is the
**single source of truth** for every number and exhibit the report quotes: it runs the statistical tests,
engineers features inside a leakage-safe Pipeline, races six model families under repeated stratified CV,
evaluates the winner at a **clinical (sensitivity-first) operating point**, states the fairness constraint,
and writes `reports/_key_figures.json` + the seven `reports/figures/ex0*.png` exhibits.

**Metric doctrine:** the positive class is **Malignant by cost** — we lead with **ROC-AUC, PR-AUC, and
malignant recall at a chosen threshold**, never plain accuracy.

In [1]:
# --- DESIGN.md palette + matplotlib style (inlined; notebooks import no local modules) ---
import matplotlib as mpl
import matplotlib.pyplot as plt

NAVY   = "#051C2C"   # ink only, never a series fill
BLUE   = "#2251FF"   # McKinsey blue -> emphasis / Malignant (at-risk class)
TEAL   = "#00857C"   # secondary series -> Benign
CYAN   = "#00A9F4"   # tertiary categorical
AMBER  = "#C1841C"   # reference / decision-threshold lines
SLATE  = "#7F93A6"   # muted labels
GREY   = "#9FADB8"   # neutral context (non-highlighted)
GRID   = "#E9ECEF"   # gridlines
CAT    = [BLUE, TEAL, AMBER, CYAN, SLATE]           # fixed categorical order
DIAG   = {"M": BLUE, "B": TEAL}                      # semantic: Malignant=Blue, Benign=Teal

def apply_style():
    mpl.rcParams.update({
        "figure.facecolor": "white", "axes.facecolor": "white", "savefig.facecolor": "white",
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "Helvetica Neue", "DejaVu Sans"],
        "font.size": 10.5, "axes.titlesize": 12.5, "axes.titleweight": "bold",
        "axes.titlelocation": "left", "axes.titlepad": 10,
        "axes.edgecolor": SLATE, "axes.linewidth": 0.8, "axes.labelcolor": NAVY,
        "axes.spines.top": False, "axes.spines.right": False,
        "text.color": NAVY, "xtick.color": NAVY, "ytick.color": NAVY,
        "grid.color": GRID, "grid.linewidth": 0.8, "axes.grid": True, "axes.grid.axis": "y",
        "legend.frameon": False, "figure.dpi": 110, "savefig.dpi": 150, "savefig.bbox": "tight",
    })

apply_style()

from pathlib import Path
import json, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

from sklearn.model_selection import (train_test_split, RepeatedStratifiedKFold, StratifiedKFold,
                                     cross_val_score, cross_val_predict)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (roc_auc_score, average_precision_score, roc_curve, precision_recall_curve,
    confusion_matrix, recall_score, precision_score, f1_score, accuracy_score)
from sklearn.calibration import calibration_curve
from scipy import stats
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.outliers_influence import variance_inflation_factor

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
FIG = ROOT / "reports" / "figures"; FIG.mkdir(parents=True, exist_ok=True)
REP = ROOT / "reports"
MOD = ROOT / "models"; MOD.mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(ROOT / "data" / "processed" / "wdbc_clean.parquet")
RAW30 = [c for c in df.columns if c not in ("diagnosis","diagnosis_binary","zero_concavity_flag")]
y = df["diagnosis_binary"].values
KF = {}  # key-figures dict written to JSON at the end
KF["n_total"], KF["n_benign"], KF["n_malignant"] = len(df), int((y==0).sum()), int((y==1).sum())
KF["pos_rate"] = round(float(y.mean()), 4)
print("analysis frame:", df.shape, "| malignant rate:", KF["pos_rate"])

analysis frame: (569, 33) | malignant rate: 0.3726


## Part A — Statistical testing (Stage 5a): which features separate the classes?

For each of the 30 features: Welch's t-test (unequal variance) **and** Mann-Whitney U (robust to the
zero-inflated, skewed features), **Cohen's d with a bootstrap 95% CI**, and univariate ROC-AUC. The 30
p-values are corrected with **Benjamini-Hochberg FDR**. Effect size is reported alongside every p —
a significant-but-tiny effect is noise, not a finding.

In [2]:
rng = np.random.default_rng(RANDOM_STATE)
def cohen_d(a, b):
    na, nb = len(a), len(b)
    sp = np.sqrt(((na-1)*a.std(ddof=1)**2 + (nb-1)*b.std(ddof=1)**2)/(na+nb-2))
    return (a.mean()-b.mean())/sp
def d_ci(a, b, n=2000):
    ds = [cohen_d(rng.choice(a,len(a)), rng.choice(b,len(b))) for _ in range(n)]
    return np.percentile(ds, [2.5, 97.5])

rows = []
for f in RAW30:
    a, b = df.loc[y==1, f].values, df.loc[y==0, f].values
    t_p = stats.ttest_ind(a, b, equal_var=False).pvalue
    u_p = stats.mannwhitneyu(a, b, alternative="two-sided").pvalue
    d = cohen_d(a, b); lo, hi = d_ci(a, b)
    rows.append({"feature": f, "cohens_d": d, "d_lo": lo, "d_hi": hi,
                 "auc": roc_auc_score(y, df[f].values), "t_p": t_p, "u_p": u_p})
stat = pd.DataFrame(rows)
stat["q_bh"] = multipletests(stat["t_p"], method="fdr_bh")[1]
stat = stat.sort_values("cohens_d", key=np.abs, ascending=False).reset_index(drop=True)
KF["n_sig_fdr"] = int((stat["q_bh"] < 0.05).sum())
print(f"features significant after BH-FDR (q<0.05): {KF['n_sig_fdr']} / 30")
stat.head(10).round(4)

features significant after BH-FDR (q<0.05): 26 / 30


,feature,cohens_d,d_lo,d_hi,auc,t_p,u_p,q_bh
0,concave points_worst,2.6926,2.4579,2.9532,0.9667,0.0,0.0,0.0
1,perimeter_worst,2.5982,2.3932,2.8455,0.9755,0.0,0.0,0.0
2,concave points_mean,2.5452,2.3195,2.8140,0.9644,0.0,0.0,0.0
3,radius_worst,2.5439,2.3500,2.7931,0.9704,0.0,0.0,0.0
4,perimeter_mean,2.2895,2.0860,2.5252,0.9469,0.0,0.0,0.0
5,area_worst,2.2302,2.0106,2.4926,0.9698,0.0,0.0,0.0
6,radius_mean,2.2055,2.0031,2.4377,0.9375,0.0,0.0,0.0
7,area_mean,2.0757,1.8740,2.3087,0.9383,0.0,0.0,0.0
8,concavity_mean,2.0033,1.7331,2.3048,0.9378,0.0,0.0,0.0
9,concavity_worst,1.8119,1.5470,2.1138,0.9214,0.0,0.0,0.0


In [3]:
# persist the full stat table + headline numbers
stat.round(6).to_csv(REP / "_stat_tests.csv", index=False)
top = stat.head(6)
KF["top_features"] = [
    {"feature": r.feature, "cohens_d": round(r.cohens_d,3),
     "ci": [round(r.d_lo,3), round(r.d_hi,3)], "auc": round(r.auc,4)}
    for r in top.itertuples()]
KF["best_single_feature"] = stat.sort_values("auc", ascending=False).iloc[0]["feature"]
KF["best_single_auc"] = round(float(stat["auc"].max()), 4)
weak = stat.reindex(stat["cohens_d"].abs().sort_values().index).head(4)["feature"].tolist()
KF["weakest_features"] = weak
print("best single feature:", KF["best_single_feature"], KF["best_single_auc"])
print("near-useless (|d|~0):", weak)

best single feature: perimeter_worst 0.9755
near-useless (|d|~0): ['symmetry_se', 'texture_se', 'fractal_dimension_mean', 'smoothness_se']


### Collinearity verdict — the 30 features are not 30 signals (VIF)

*So What:* Sky-high VIFs confirm the by-construction redundancy from EDA. This is *why* the linear models
below use regularization and why single-feature importances are unstable — the same signal is spread across
correlated columns.

In [4]:
Xs = StandardScaler().fit_transform(df[RAW30].values)
vif = pd.Series({RAW30[i]: variance_inflation_factor(Xs, i) for i in range(len(RAW30))})
corr = df[RAW30].corr().abs()
n_hi = int(((corr > 0.9) & (corr < 1.0)).values.sum() // 2)
KF["vif_max"] = round(float(vif.max()), 1)
KF["vif_over10"] = int((vif > 10).sum())
KF["collinear_pairs_over_090"] = n_hi
print(f"max VIF: {KF['vif_max']:.0f} | features with VIF>10: {KF['vif_over10']}/30 | |r|>0.9 pairs: {n_hi}")
print("worst offenders:\n", vif.sort_values(ascending=False).head(5).round(0))

max VIF: 3806 | features with VIF>10: 23/30 | |r|>0.9 pairs: 21
worst offenders:
 radius_mean        3806.0
perimeter_mean     3786.0
radius_worst        799.0
perimeter_worst     405.0
area_mean           348.0
dtype: float64


## Stage 4 — Feature engineering (leakage-safe)

Five domain features from the data notes §7, each **row-wise deterministic** (ratios/sums) so they can be
computed before the split without leakage; **StandardScaler and any PCA are fit inside the Pipeline on
train folds only**. Trees see the full matrix; the linear/SVM models get it scaled.

In [5]:
def engineer(frame):
    out = frame[RAW30].copy()
    out["shape_perim_per_radius"] = frame["perimeter_mean"]/frame["radius_mean"]
    out["shape_area_per_radius2"] = frame["area_mean"]/(frame["radius_mean"]**2)
    out["radius_cv"]              = frame["radius_se"]/frame["radius_mean"]
    out["radius_worst_to_mean"]   = frame["radius_worst"]/frame["radius_mean"]
    out["irregularity_index"]     = frame["concavity_mean"]+frame["concave points_mean"]+frame["fractal_dimension_mean"]
    return out

X = engineer(df)
KF["n_features_model"] = X.shape[1]
print("model matrix:", X.shape, "(30 raw + 5 engineered)")

# Held-out test set touched ONCE, at the very end. Stratified 80/20.
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE)
KF["n_train"], KF["n_test"] = len(X_tr), len(X_te)
print("train:", X_tr.shape, "test:", X_te.shape, "| test malignant:", int(y_te.sum()))

model matrix: (569, 35) (30 raw + 5 engineered)
train: (455, 35) test: (114, 35) | test malignant: 42


## Stage 5b — Model bake-off (simple → complex)

Two baselines everything must beat (majority-class, and a **single-feature** logistic), then five real
models. Selection is by **RepeatedStratifiedKFold (10×3 = 30 fits)** ROC-AUC on the **training set only**;
the held-out test is scored once afterwards.

In [6]:
cv = RepeatedStratifiedKFold(n_splits=10, n_repeats=3, random_state=RANDOM_STATE)
scaler = lambda est: Pipeline([("sc", StandardScaler()), ("clf", est)])
best_feat = KF["best_single_feature"]

models = {
    "Majority baseline":     DummyClassifier(strategy="most_frequent"),
    f"1-feature logistic":   Pipeline([("sc", StandardScaler()),
                                       ("clf", LogisticRegression(max_iter=5000))]),  # fit on 1 col below
    "Logistic (L2)":         scaler(LogisticRegression(penalty="l2", C=1.0, max_iter=5000)),
    "Logistic (L1)":         scaler(LogisticRegression(penalty="l1", solver="liblinear", C=1.0, max_iter=5000)),
    "SVM (RBF)":             scaler(SVC(kernel="rbf", C=1.0, probability=True, random_state=RANDOM_STATE)),
    "Random Forest":         RandomForestClassifier(n_estimators=400, random_state=RANDOM_STATE, n_jobs=-1),
    "Gradient Boosting":     GradientBoostingClassifier(random_state=RANDOM_STATE),
}

def cv_auc(name, est):
    if name == "1-feature logistic":
        return cross_val_score(est, X_tr[[best_feat]], y_tr, cv=cv, scoring="roc_auc", n_jobs=-1)
    if name == "Majority baseline":
        return cross_val_score(est, X_tr, y_tr, cv=cv, scoring="roc_auc", n_jobs=-1)
    return cross_val_score(est, X_tr, y_tr, cv=cv, scoring="roc_auc", n_jobs=-1)

results = []
for name, est in models.items():
    s = cv_auc(name, est)
    results.append({"model": name, "cv_auc_mean": s.mean(), "cv_auc_std": s.std()})
    print(f"{name:22s}  CV ROC-AUC = {s.mean():.4f} ± {s.std():.4f}")
res = pd.DataFrame(results)

Majority baseline       CV ROC-AUC = 0.5000 ± 0.0000


1-feature logistic      CV ROC-AUC = 0.9756 ± 0.0188
Logistic (L2)           CV ROC-AUC = 0.9963 ± 0.0056
Logistic (L1)           CV ROC-AUC = 0.9928 ± 0.0096


SVM (RBF)               CV ROC-AUC = 0.9961 ± 0.0059


Random Forest           CV ROC-AUC = 0.9903 ± 0.0135


Gradient Boosting       CV ROC-AUC = 0.9895 ± 0.0152


### Fit on train, score the held-out test **once** — full metric panel

Accuracy is shown but *not* used for selection. The clinically decisive columns are **Recall (malignant
sensitivity)** and **PR-AUC**.

In [7]:
def fit_predict(name, est):
    cols = [best_feat] if name == "1-feature logistic" else X.columns.tolist()
    est.fit(X_tr[cols], y_tr)
    if hasattr(est, "predict_proba"):
        proba = est.predict_proba(X_te[cols])[:, 1]
    elif hasattr(est, "decision_function"):
        d = est.decision_function(X_te[cols]); proba = (d - d.min())/(d.max()-d.min())
    else:
        proba = est.predict(X_te[cols]).astype(float)
    pred = (proba >= 0.5).astype(int)
    return est, proba, pred

fitted, panel = {}, []
for name, est in models.items():
    est, proba, pred = fit_predict(name, est)
    fitted[name] = (est, proba)
    panel.append({"model": name,
        "test_auc": roc_auc_score(y_te, proba) if name!="Majority baseline" else 0.5,
        "test_pr_auc": average_precision_score(y_te, proba),
        "recall_M": recall_score(y_te, pred), "precision_M": precision_score(y_te, pred, zero_division=0),
        "specificity": recall_score(y_te, pred, pos_label=0), "f1": f1_score(y_te, pred),
        "accuracy": accuracy_score(y_te, pred)})
panel = pd.DataFrame(panel).merge(res, on="model")
panel = panel[["model","cv_auc_mean","cv_auc_std","test_auc","test_pr_auc","recall_M","precision_M","specificity","f1","accuracy"]]
panel.round(4)

,model,cv_auc_mean,cv_auc_std,test_auc,test_pr_auc,recall_M,precision_M,specificity,f1,accuracy
0,Majority baseline,0.5000,0.0000,0.5000,0.3684,0.0000,0.0000,1.0000,0.0000,0.6316
1,1-feature logistic,0.9756,0.0188,0.9874,0.9781,0.7381,0.9394,0.9722,0.8267,0.8860
2,Logistic (L2),0.9963,0.0056,0.9977,0.9963,0.9524,0.9756,0.9861,0.9639,0.9737
3,Logistic (L1),0.9928,0.0096,0.9977,0.9963,0.9762,0.9762,0.9861,0.9762,0.9825
4,SVM (RBF),0.9961,0.0059,0.9964,0.9942,0.9286,0.9750,0.9861,0.9512,0.9649
5,Random Forest,0.9903,0.0135,0.9977,0.9962,0.9286,1.0000,1.0000,0.9630,0.9737
6,Gradient Boosting,0.9895,0.0152,0.9967,0.9954,0.9048,1.0000,1.0000,0.9500,0.9649


In [8]:
# choose winner: exclude baselines, best CV AUC; prefer the simpler model within 0.005
cand = panel[~panel["model"].isin(["Majority baseline","1-feature logistic"])].copy()
best_cv = cand["cv_auc_mean"].max()
near = cand[cand["cv_auc_mean"] >= best_cv - 0.005]
SIMPLICITY = ["Logistic (L2)","Logistic (L1)","SVM (RBF)","Random Forest","Gradient Boosting"]
winner = sorted(near["model"], key=lambda m: SIMPLICITY.index(m))[0]
KF["winner"] = winner
wrow = panel[panel["model"]==winner].iloc[0]
KF["winner_cv_auc"] = round(float(wrow["cv_auc_mean"]),4); KF["winner_cv_std"]=round(float(wrow["cv_auc_std"]),4)
KF["winner_test_auc"] = round(float(wrow["test_auc"]),4); KF["winner_pr_auc"]=round(float(wrow["test_pr_auc"]),4)
KF["baseline_cv_auc"] = round(float(panel[panel.model=="Majority baseline"]["cv_auc_mean"].iloc[0]),4)
KF["onefeat_cv_auc"]  = round(float(panel[panel.model=="1-feature logistic"]["cv_auc_mean"].iloc[0]),4)
panel.round(4).to_csv(REP / "_model_results.csv", index=False)
print(f"WINNER (simplest within noise of best CV AUC): {winner}")
print(f"  CV AUC {KF['winner_cv_auc']}±{KF['winner_cv_std']} | test AUC {KF['winner_test_auc']} | PR-AUC {KF['winner_pr_auc']}")

WINNER (simplest within noise of best CV AUC): Logistic (L2)
  CV AUC 0.9963±0.0056 | test AUC 0.9977 | PR-AUC 0.9963


## Stage 6 — Evaluation at a clinical operating point

Default 0.5 is not the right threshold when a false negative (missed malignancy) is the costly error. We
pick the threshold that delivers **≥ 98% malignant sensitivity**, chosen on **cross-validated training
predictions** (never on the test set), then report the held-out confusion matrix at that threshold in raw
counts.

In [9]:
win_est = fitted[winner][0]
cols = X.columns.tolist()
# CV probabilities on TRAIN to pick threshold (no test leakage).
# Use a single StratifiedKFold partition (cross_val_predict requires each row predicted once).
cv_thr = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)
cv_proba_tr = cross_val_predict(win_est, X_tr[cols], y_tr, cv=cv_thr, method="predict_proba", n_jobs=-1)[:,1]
TARGET_SENS = 0.98
prec, rec, thr = precision_recall_curve(y_tr, cv_proba_tr)
ok = np.where(rec[:-1] >= TARGET_SENS)[0]
op_thr = float(thr[ok[-1]]) if len(ok) else 0.5   # highest threshold still meeting sensitivity
KF["target_sensitivity"] = TARGET_SENS
KF["operating_threshold"] = round(op_thr, 4)

te_proba = fitted[winner][1]
pred_op = (te_proba >= op_thr).astype(int)
tn, fp, fn, tp = confusion_matrix(y_te, pred_op).ravel()
KF["cm"] = {"tp":int(tp),"fp":int(fp),"tn":int(tn),"fn":int(fn)}
KF["op_sensitivity"] = round(tp/(tp+fn), 4)
KF["op_specificity"] = round(tn/(tn+fp), 4)
KF["op_precision"]   = round(tp/(tp+fp), 4)
print(f"operating threshold = {op_thr:.3f}")
print(f"held-out test @ threshold: TP={tp} FN={fn} FP={fp} TN={tn}")
print(f"  malignant sensitivity = {KF['op_sensitivity']:.3f}  (missed {fn}/{tp+fn} malignancies)")
print(f"  specificity = {KF['op_specificity']:.3f}  precision = {KF['op_precision']:.3f}")

operating threshold = 0.154
held-out test @ threshold: TP=41 FN=1 FP=4 TN=68
  malignant sensitivity = 0.976  (missed 1/42 malignancies)
  specificity = 0.944  precision = 0.911


### Calibration & permutation importance — the collinearity signature

A clinical score must be **calibrated** (a "0.8" should mean ~80% malignant) — we report the **Brier
score**. Permutation importance tells a subtler story: because the size/irregularity features are nearly
redundant, shuffling any *one* barely moves AUC (the others substitute). So the importances are **tiny and
dispersed** — the finding is not "one feature rules" but "**no single measurement is irreplaceable**", the
model-side echo of the VIF result. The trustworthy driver ranking remains the univariate Cohen's d.

In [10]:
from sklearn.metrics import brier_score_loss
# calibration on test
frac_pos, mean_pred = calibration_curve(y_te, te_proba, n_bins=8, strategy="quantile")
KF["brier"] = round(float(brier_score_loss(y_te, te_proba)), 4)
# permutation importance on test (SHAP intentionally avoided — keeps clone dependency-light)
perm = permutation_importance(win_est, X_te[cols], y_te, scoring="roc_auc",
                              n_repeats=30, random_state=RANDOM_STATE, n_jobs=1)
imp = pd.Series(perm.importances_mean, index=cols).sort_values(ascending=False)
KF["top_importance"] = [{"feature": f, "importance": round(float(imp[f]),4)} for f in imp.head(6).index]
KF["importance_max"] = round(float(imp.max()), 4)
top_d = set(stat.head(6)["feature"])
KF["importance_topd_overlap"] = len(set(imp.head(6).index) & top_d)
print(f"Brier score (test): {KF['brier']}  (lower is better; 0 = perfect)")
print("top permutation-importance features (note the tiny magnitudes):\n", imp.head(6).round(4))
print(f"\nmax single-feature importance = {KF['importance_max']} AUC -> no feature is irreplaceable (collinearity)")

Brier score (test): 0.0209  (lower is better; 0 = perfect)
top permutation-importance features (note the tiny magnitudes):
 symmetry_worst          0.0098
texture_worst           0.0064
concavity_worst         0.0055
concave points_worst    0.0033
concave points_mean     0.0031
irregularity_index      0.0025
dtype: float64

max single-feature importance = 0.0098 AUC -> no feature is irreplaceable (collinearity)


## Cross-cutting — the fairness audit that cannot be run

The model **affects people**, so STRUCTURE.md requires a fairness audit. But the WDBC data carries **no age,
race, site, or any demographic attribute** — so subgroup performance **cannot be measured**. That is not a
free pass; it is a **hard limitation that caps the deployment claim**: the model is validated on a single
historical Wisconsin sample, and its behaviour across populations, instruments, and labs is *unknown and
un-auditable from this data*. The recommendation is therefore capped at **second-reader use with mandatory
prospective multi-site validation before any autonomous role.**

In [11]:
KF["fairness"] = ("No demographic attributes exist in WDBC (age/race/site absent), so subgroup performance "
                  "cannot be audited. Deployment claim capped at second-reader use pending prospective "
                  "multi-site validation.")
KF["limitations"] = [
    "Single-source historical sample (n=569, Wisconsin); no external validation cohort.",
    "No demographics -> fairness subgroup audit impossible (see above).",
    "'Worst' features may be instrument/segmentation-specific; portability unproven.",
    "Diagnostic/association only -> decision-support, never autonomous diagnosis.",
    "Near-perfect separability is a known property of this benchmark; real-world screens are harder.",
]
print(KF["fairness"])

No demographic attributes exist in WDBC (age/race/site absent), so subgroup performance cannot be audited. Deployment claim capped at second-reader use pending prospective multi-site validation.


## Exhibits — write the seven report figures to `reports/figures/`

In [12]:
import matplotlib.pyplot as plt
import matplotlib.patches as mp
from sklearn.decomposition import PCA

def base_of(f):
    for b in ["radius","texture","perimeter","area","smoothness","compactness",
              "concavity","concave points","symmetry","fractal_dimension"]:
        if f.startswith(b): return b

# EX1 — Cohen's d ranking
s12 = stat.head(12).iloc[::-1]
cols_c = [BLUE if base_of(f) in ("radius","perimeter","area") else
          (AMBER if base_of(f) in ("compactness","concavity","concave points") else GREY) for f in s12["feature"]]
fig, ax = plt.subplots(figsize=(7.6,5))
ax.barh(range(len(s12)), s12["cohens_d"], color=cols_c)
ax.set_yticks(range(len(s12))); ax.set_yticklabels(s12["feature"], fontsize=9)
ax.axvline(0, color=SLATE, lw=.8)
ax.set_title("Malignant nuclei separate on size and irregularity — not texture", fontsize=12)
ax.set_xlabel("Cohen's d (malignant − benign, SD units)")
ax.legend(handles=[mp.Patch(color=BLUE,label="Size"),mp.Patch(color=AMBER,label="Irregularity"),
                   mp.Patch(color=GREY,label="Texture/form")], loc="lower right", fontsize=8)
fig.savefig(FIG/"ex01_cohend.png"); plt.close(fig)

# EX2 — collinearity heatmap
order = [f"{b}_{s}" for s in ["mean","se","worst"] for b in
         ["radius","texture","perimeter","area","smoothness","compactness","concavity","concave points","symmetry","fractal_dimension"]]
cm = df[RAW30].corr().loc[order, order]
fig, ax = plt.subplots(figsize=(8.2,7))
im = ax.imshow(cm.values, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(30)); ax.set_xticklabels(order, rotation=90, fontsize=5.5)
ax.set_yticks(range(30)); ax.set_yticklabels(order, fontsize=5.5)
ax.set_title("The 30 features are only a few real signals", fontsize=12)
fig.colorbar(im, ax=ax, shrink=.7, label="|Pearson r|→ signed")
fig.savefig(FIG/"ex02_corr.png"); plt.close(fig)

# EX3 — univariate AUC
au = stat.set_index("feature")["auc"].sort_values(ascending=False).head(10).iloc[::-1]
fig, ax = plt.subplots(figsize=(7.4,4.6))
ax.barh(range(len(au)), au.values, color=[BLUE if v>=.9 else GREY for v in au.values])
ax.set_yticks(range(len(au))); ax.set_yticklabels(au.index, fontsize=9); ax.set_xlim(.5,1)
ax.axvline(.5,color=SLATE,ls=":",lw=.8)
for i,v in enumerate(au.values): ax.text(v+.004,i,f"{v:.3f}",va="center",fontsize=8)
ax.set_title("One feature already reaches AUC ≈ 0.98", fontsize=12)
ax.set_xlabel("univariate ROC-AUC vs diagnosis")
fig.savefig(FIG/"ex03_univariate_auc.png"); plt.close(fig)

# EX4 — PCA
Xall = StandardScaler().fit_transform(df[RAW30].values)
pca = PCA(random_state=RANDOM_STATE).fit(Xall); Z = pca.transform(Xall); ev = pca.explained_variance_ratio_
KF["pca_pc1"]=round(float(ev[0]*100),1); KF["pca_pc12"]=round(float(ev[:2].sum()*100),1); KF["pca_pc123"]=round(float(ev[:3].sum()*100),1)
fig, axes = plt.subplots(1,2,figsize=(11,4.3))
axes[0].bar(range(1,11), ev[:10]*100, color=GREY)
axes[0].plot(range(1,11), np.cumsum(ev[:10])*100, color=BLUE, marker="o", ms=4)
axes[0].set_title(f"{KF['pca_pc12']:.0f}% of variance in 2 components", fontsize=11.5)
axes[0].set_xlabel("component"); axes[0].set_ylabel("% variance / cumulative")
for c,col,l in [(0,TEAL,"Benign"),(1,BLUE,"Malignant")]:
    m=y==c; axes[1].scatter(Z[m,0],Z[m,1],s=12,c=col,alpha=.6,label=l,edgecolors="none")
axes[1].set_title("PC1 alone almost splits the classes", fontsize=11.5)
axes[1].set_xlabel(f"PC1 ({ev[0]*100:.0f}%)"); axes[1].set_ylabel(f"PC2 ({ev[1]*100:.0f}%)"); axes[1].legend()
fig.savefig(FIG/"ex04_pca.png"); plt.close(fig)
print("EX1-4 saved | PCA:", KF["pca_pc1"], KF["pca_pc12"], KF["pca_pc123"])

EX1-4 saved | PCA: 44.3 63.2 72.6


In [13]:
# EX5 — ROC + PR curves for the real models
real = ["Logistic (L2)","Logistic (L1)","SVM (RBF)","Random Forest","Gradient Boosting"]
palette = dict(zip(real, [BLUE, CYAN, AMBER, TEAL, SLATE]))
fig, (a1,a2) = plt.subplots(1,2,figsize=(11,4.6))
for name in real:
    _, proba = fitted[name]
    fpr,tpr,_ = roc_curve(y_te, proba); a1.plot(fpr,tpr,color=palette[name],lw=2 if name==winner else 1.2,
        label=f"{name} ({roc_auc_score(y_te,proba):.3f})")
    pr,rc,_ = precision_recall_curve(y_te, proba); a2.plot(rc,pr,color=palette[name],lw=2 if name==winner else 1.2,
        label=f"{name} ({average_precision_score(y_te,proba):.3f})")
a1.plot([0,1],[0,1],color=GREY,ls=":",lw=.8); a1.set_title("Every model clears ROC-AUC ≈ 0.98", fontsize=11.5)
a1.set_xlabel("false positive rate"); a1.set_ylabel("true positive rate"); a1.legend(fontsize=7.5, loc="lower right")
a2.set_title("PR-AUC: the choice is the operating point, not the algorithm", fontsize=11.5)
a2.set_xlabel("recall (malignant)"); a2.set_ylabel("precision"); a2.legend(fontsize=7.5, loc="lower left")
fig.savefig(FIG/"ex05_roc_pr.png"); plt.close(fig)

# EX6 — confusion matrix at operating point
cmat = np.array([[tn,fp],[fn,tp]])
fig, ax = plt.subplots(figsize=(5.2,4.4))
im = ax.imshow(cmat, cmap="Blues")
for i in range(2):
    for j in range(2):
        ax.text(j,i,cmat[i,j],ha="center",va="center",fontsize=20,
                color="white" if cmat[i,j]>cmat.max()/2 else NAVY)
ax.set_xticks([0,1]); ax.set_xticklabels(["Pred B","Pred M"])
ax.set_yticks([0,1]); ax.set_yticklabels(["Actual B","Actual M"])
ax.set_title(f"At the clinical threshold: {fn} missed malignancy, {fp} false alarms", fontsize=11)
ax.grid(False)
fig.savefig(FIG/"ex06_confusion.png"); plt.close(fig)

# EX7 — permutation importance, colored by feature family (collinearity dispersion story)
def fam(f):
    b = base_of(f) if base_of(f) else ""
    if b in ("radius","perimeter","area") or f.startswith(("shape_","radius_")): return "Size"
    if b in ("compactness","concavity","concave points") or f=="irregularity_index": return "Irregularity"
    return "Texture/form"
FAMC = {"Size":BLUE,"Irregularity":AMBER,"Texture/form":GREY}
iv = imp.head(8).iloc[::-1]
fig, ax = plt.subplots(figsize=(7.4,4.8))
ax.barh(range(len(iv)), iv.values, color=[FAMC[fam(f)] for f in iv.index])
ax.set_yticks(range(len(iv))); ax.set_yticklabels(iv.index, fontsize=9)
ax.set_title("No single measurement is irreplaceable — importance disperses across collinear features",
             fontsize=10.5)
ax.set_xlabel("permutation importance (Δ ROC-AUC when shuffled; note the small scale)")
ax.legend(handles=[mp.Patch(color=BLUE,label="Size"),mp.Patch(color=AMBER,label="Irregularity"),
                   mp.Patch(color=GREY,label="Texture/form")], loc="lower right", fontsize=8)
fig.savefig(FIG/"ex07_importance.png"); plt.close(fig)
print("EX5-7 saved")

EX5-7 saved


## Persist key figures + model card

In [14]:
import joblib
KF["cv_scheme"] = "RepeatedStratifiedKFold(10x3=30 fits) for selection; 20% held-out test scored once"
with open(REP / "_key_figures.json", "w") as f:
    json.dump(KF, f, indent=2)
joblib.dump(fitted[winner][0], MOD / "winning_model.pkl")
card = {"model": winner, "features": cols, "cv_auc": KF["winner_cv_auc"], "test_auc": KF["winner_test_auc"],
        "pr_auc": KF["winner_pr_auc"], "operating_threshold": KF["operating_threshold"],
        "op_sensitivity": KF["op_sensitivity"], "op_specificity": KF["op_specificity"],
        "positive_class": "Malignant (1)", "seed": RANDOM_STATE, "intended_use": "second-reader decision support only"}
with open(MOD / "model_card.json", "w") as f:
    json.dump(card, f, indent=2)
print("wrote reports/_key_figures.json, models/winning_model.pkl, models/model_card.json")
print(json.dumps({k:KF[k] for k in ["winner","winner_cv_auc","winner_test_auc","winner_pr_auc",
      "operating_threshold","op_sensitivity","op_specificity","cm"]}, indent=2))

wrote reports/_key_figures.json, models/winning_model.pkl, models/model_card.json
{
  "winner": "Logistic (L2)",
  "winner_cv_auc": 0.9963,
  "winner_test_auc": 0.9977,
  "winner_pr_auc": 0.9963,
  "operating_threshold": 0.1538,
  "op_sensitivity": 0.9762,
  "op_specificity": 0.9444,
  "cm": {
    "tp": 41,
    "fp": 4,
    "tn": 68,
    "fn": 1
  }
}


## Gate Checklists

**Stage 5a — Statistical testing**
- [x] Tests chosen for the data — Welch **and** Mann-Whitney (skew/zero-inflation), not convenience
- [x] Assumptions handled — non-parametric alongside parametric; effect size + bootstrap CI reported
- [x] Effect sizes alongside p — Cohen's d with 95% CI for every feature
- [x] Multiple-comparison correction — Benjamini-Hochberg FDR across 30 tests
- [x] Interpreted in clinical terms — separation framed by feature family, not r-units

**Stage 4 — Feature engineering**
- [x] Rationale documented (5 domain features from data notes §7)
- [x] No leakage — row-wise transforms pre-split; scaling/PCA fit inside the Pipeline on train folds
- [x] Importance analysis completed (permutation importance)
- [x] Reproducible — deterministic, seeded, pipeline-ready

**Stage 6 — Evaluation**
- [x] Held-out test scored once; selection on CV only
- [x] Multiple metrics — ROC-AUC, PR-AUC, recall, precision, specificity, F1 (accuracy shown, not decisive)
- [x] Beats baselines — majority-class and single-feature both cleared
- [x] Business impact quantified — confusion matrix at a clinical operating point, in raw counts
- [x] Limitations documented; calibration checked
- [x] Reproducible — seed 42, 30-fit CV logged

**Cross-cutting**
- [x] Governance carried from nb 01; fairness constraint (no demographics) surfaced in the main narrative
- [x] Deployment claim capped accordingly

**→ Proceed to `05_reporting.ipynb`.**